# Etapa 2 - Análise Exploratória
Projeto Banco do Brasil + ODS 8

Este notebook lê os seis CSVs oficiais do Banco Central, consolida a base, analisa qualidade, estatísticas descritivas e visualizações.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'dados').exists():
    ROOT = ROOT.parent.parent
RAW = ROOT / 'dados' / 'raw'
ARQUIVOS = sorted(RAW.glob('ranking_bcb_20*_T*.csv'))
len(ARQUIVOS), [p.name for p in ARQUIVOS]

## 1. Leitura e consolidação

In [ ]:
quadros = []
for arquivo in ARQUIVOS:
    df = pd.read_csv(arquivo, sep=';', encoding='utf-8-sig', dtype=str)
    df = df.loc[:, [c for c in df.columns if str(c).strip()]]
    df['arquivo_origem'] = arquivo.name
    quadros.append(df)
base = pd.concat(quadros, ignore_index=True, sort=False)
base.shape

## 2. Metadados, nulos e tipos

In [ ]:
pd.DataFrame({
    'tipo': base.dtypes.astype(str),
    'nulos': base.isna().sum(),
    'valores_distintos': base.nunique(dropna=True)
})

## 3. Conversão das variáveis numéricas

In [ ]:
def numero_br(valor):
    if pd.isna(valor): return np.nan
    texto = str(valor).strip()
    if not texto: return np.nan
    if ',' in texto and '.' in texto:
        texto = texto.replace('.', '').replace(',', '.')
    elif ',' in texto:
        texto = texto.replace(',', '.')
    texto = re.sub(r'[^0-9.\-]', '', texto)
    try: return float(texto)
    except ValueError: return np.nan

numericas = [
    'Índice','Quantidade total de reclamações respondidas',
    'Quantidade de reclamações procedentes',
    'Quantidade de reclamações procedentes extrapoladas',
    'Quantidade total de reclamações analisadas',
    'Quantidade total de clientes – CCS e SCR'
]
for c in numericas:
    base[c] = base[c].map(numero_br)
base[numericas].describe().T

## 4. Recorte do Banco do Brasil

In [ ]:
bb = base[base['Instituição financeira'].eq('BB (conglomerado)')].copy()
bb[['Ano','Trimestre','Índice','Quantidade de reclamações procedentes extrapoladas','Quantidade total de clientes – CCS e SCR']]

## 5. Estatísticas descritivas do BB

In [ ]:
bb[numericas].describe().T

## 6. Gráfico do índice

In [ ]:
bb['Periodo'] = bb['Ano'].astype(str) + ' T' + bb['Trimestre'].str.extract(r'([1-4])')[0]
plt.figure(figsize=(9,5))
plt.plot(bb['Periodo'], bb['Índice'], marker='o')
plt.title('Evolução do índice de reclamações do Banco do Brasil')
plt.xlabel('Período')
plt.ylabel('Índice de reclamações')
plt.grid(axis='y', alpha=.25)
plt.show()

## 7. Interpretação
A interpretação deve considerar a metodologia do Banco Central. Valores extremos entre instituições não são tratados automaticamente como erro, pois a base reúne instituições de portes e perfis muito diferentes.